In [ ]:
import pandas as pd
from data_tools.db import Manager

engine = Manager("mvh-admin", "mvh").engine

In [ ]:
# tbl_lakhely_szekhely_2024 = create_lakhely_szekhely_tbl(2024, engine)
# tbl_lakhely_szekhely_2026 = create_lakhely_szekhely_tbl(2026, engine)

In [ ]:
hasznositasi_kodtar_2024 = pd.read_excel(
    "input/hasznositasi_kodtar_2024.xlsx", engine="calamine"
)
hasznositasi_kodtar_2025 = pd.read_excel(
    "input/hasznositasi_kodtar_2025.xlsx", engine="calamine"
)
hasznositasi_kodtar_2026 = pd.read_excel(
    "input/hasznositasi_kodtar_2026.xlsx", engine="calamine"
)

In [ ]:
SQL = """SELECT regszam
FROM altalanos.ugyfela
WHERE ev = 2026
  AND vallalkozasi_forma IN (
      'Kft.',
      'Rt.',
      'Agrárgazd.-i szöv.',
      'Bt.',
      'Nonprofit Kft.',
      'Erdőbirt. társulat',
      'Nonprofit Rt.',
      'Egyéni cég'
  );"""

tbl_tarsas = pd.read_sql(SQL, con=engine)

data = pd.read_sql(
    "select ev, regszam, hkod, terulet_elf, fiatal_mgi_termelo, folias_termesztes from ek_adatok.tera where ev=2026 and teruletalapu_alaptamogatas = 1 and tam_nem_ig = 0",
    con=engine,
)

data = data[data["regszam"].isin(tbl_tarsas["regszam"])]

data_helper = pd.read_parquet("input/farms_under_one_control_2026.parquet")

# data_helper[data_helper["nev"].str.contains("néhai", case=False, na=False)]
data_helper = data_helper[["lakhely_szekhely", "regszam"]]


data2 = data.merge(
    data_helper[["regszam", "lakhely_szekhely"]],
    on="regszam",
    how="left",
)

mask = data2["lakhely_szekhely"].isna()

missing_regszam = data2.loc[mask, "regszam"].drop_duplicates()

unknown_map = {
    regszam: f"Ismeretlen{i}" for i, regszam in enumerate(missing_regszam, start=1)
}

data2.loc[mask, "lakhely_szekhely"] = data2.loc[mask, "regszam"].map(unknown_map)

In [ ]:
data_merged = data2.groupby(
    ["ev", "lakhely_szekhely", "hkod", "folias_termesztes"],
    as_index=False,
    dropna=False,
).agg(
    terulet_elf=("terulet_elf", "sum"),
)

data_merged["regszam_osszevont"] = (
    pd.factorize(data_merged["lakhely_szekhely"], sort=True)[0] + 1
)

In [ ]:
def novenycsoport_osszefoglalo_pandas(
    tbl: pd.DataFrame,
    ev: int,
    hkodok: list[str],
    meretkuszob: float,
) -> pd.DataFrame:
    df = tbl.loc[
        (tbl["ev"] == ev)
        & (tbl["hkod"].isin(hkodok))
        & (tbl["folias_termesztes"] != 1)
        & (tbl["terulet_elf"] > 0)
    ].copy()

    regszam_szint = df.groupby("regszam_osszevont", as_index=False).agg(
        terulet_osszes=("terulet_elf", "sum"),
    )

    kuszob_felett = regszam_szint["terulet_osszes"] >= meretkuszob

    osszes_gazdasag = len(regszam_szint)
    kuszob_feletti_gazdasagok = kuszob_felett.sum()

    return pd.DataFrame(
        {
            "Összes terület": [regszam_szint["terulet_osszes"].sum()],
            "Összes gazdaság (db)": [osszes_gazdasag],
            "Méretküszöb feletti gazdaságok (db)": [kuszob_feletti_gazdasagok],
            "Méretküszöb feletti terület": [
                regszam_szint.loc[
                    kuszob_felett,
                    "terulet_osszes",
                ].sum()
            ],
            "Küszöb felettiek részaránya üzemekből (%)": [
                (
                    100 * kuszob_feletti_gazdasagok / osszes_gazdasag
                    if osszes_gazdasag > 0
                    else float("nan")
                )
            ],
        }
    )

In [ ]:
szanto_zoldseg_2026 = hasznositasi_kodtar_2026.loc[
    hasznositasi_kodtar_2026["Kockázatkezelési rendszer I. pillér kategória"]
    == "Szántóföldi zöldség",
    "Hasznosítási kódok 2026",
]


szanto_zoldseg_2026_lst = szanto_zoldseg_2026.tolist()

szanto_2026 = hasznositasi_kodtar_2026.loc[
    hasznositasi_kodtar_2026["AÖP szerint szántó/állandó kultúra/gyep"] == "szántó",
    "Hasznosítási kódok 2026",
]

szanto_2026_lst = szanto_2026.tolist()

In [ ]:
SQL = """
select regszam, letszam_ig from mvh_tk_kozv.tam_allat
where allat_tip = 'tejhasznú tehén' and letszam_ig > 0 and ev = 2026;
"""
data_tejelo = pd.read_sql(SQL, con=engine)

data_tejelo = data_tejelo[data_tejelo["regszam"].isin(tbl_tarsas["regszam"])]

data_tej = data_tejelo.merge(
    data_helper[["regszam", "lakhely_szekhely"]],
    on="regszam",
    how="left",
)

mask = data_tej["lakhely_szekhely"].isna()

missing_regszam = data_tej.loc[mask, "regszam"].drop_duplicates()

unknown_map = {
    regszam: f"Ismeretlen{i}" for i, regszam in enumerate(missing_regszam, start=1)
}

data_tej.loc[mask, "lakhely_szekhely"] = data_tej.loc[mask, "regszam"].map(unknown_map)

data_tej = data_tej.groupby(
    ["lakhely_szekhely"],
    as_index=False,
    dropna=False,
).agg(
    letszam_ig=("letszam_ig", "sum"),
)

data_tej["regszam_osszevont"] = (
    pd.factorize(data_tej["lakhely_szekhely"], sort=True)[0] + 1
)


def allatcsoport_osszefoglalo_pandas(
    tbl: pd.DataFrame,
    meretkuszob: float,
) -> pd.DataFrame:
    regszam_szint = (
        tbl.loc[tbl["letszam_ig"] > 0]
        .groupby("regszam_osszevont", as_index=False)
        .agg(
            letszam_osszes=("letszam_ig", "sum"),
        )
    )

    kuszob_felett = regszam_szint["letszam_osszes"] >= meretkuszob

    osszes_gazdasag = len(regszam_szint)
    kuszob_feletti_gazdasagok = kuszob_felett.sum()

    return pd.DataFrame(
        {
            "Összes létszám": [regszam_szint["letszam_osszes"].sum()],
            "Összes gazdaság (db)": [osszes_gazdasag],
            "Méretküszöb feletti gazdaságok (db)": [kuszob_feletti_gazdasagok],
            "Méretküszöb feletti létszám": [
                regszam_szint.loc[
                    kuszob_felett,
                    "letszam_osszes",
                ].sum()
            ],
            "Küszöb felettiek részaránya üzemekből (%)": [
                (
                    100 * kuszob_feletti_gazdasagok / osszes_gazdasag
                    if osszes_gazdasag > 0
                    else float("nan")
                )
            ],
        }
    )

In [ ]:
pd.set_option(
    "display.float_format",
    lambda x: f"{x:,.2f}".replace(",", " "),
)

In [ ]:
# result = novenycsoport_osszefoglalo_pandas(
#     tbl=data_merged,
#     ev=2026,
#     hkodok=szanto_2026_lst,
#     meretkuszob=115,
# )
# result = novenycsoport_osszefoglalo_pandas(
#     tbl=data_merged,
#     ev=2026,
#     hkodok=szanto_zoldseg_2026_lst,
#     meretkuszob=5,
# )
# result = novenycsoport_osszefoglalo_pandas(
#     tbl=data_merged,
#     ev=2026,
#     hkodok=["ULT01"],
#     meretkuszob=10,
# )

# result = allatcsoport_osszefoglalo_pandas(data_tej, 65)
result = allatcsoport_osszefoglalo_pandas(data_tej, 100)
# result = allatcsoport_osszefoglalo_pandas(data_tej, 150)

In [ ]:
result.to_clipboard()